Menambahkan kolom audio ke dataframe.

In [12]:
import pandas as pd
import os
import shutil
import fpdf
from fpdf import FPDF
import pdfplumber
import math
import soundfile as sf
import glob
import wave
import struct
import math
import csv
from pathlib import Path
from math import ceil
import random
# import pydub
# from pydub import AudioSegment
root = "../data/sampled"
sampled2_root = '../data/sampled2'

In [16]:
import os
import math
import pandas as pd
from fpdf import FPDF

def csv_to_pdf(file_csv, delim, output_pdf, col_widths, output_dir=None):
    '''
    purpose :
        function csv_to_pdf digunakan untuk mengkonversi file CSV menjadi file PDF berbentuk tabel.
    input :
        file_csv   : str  = input file berupa path atau nama file csv bahan konversi
        delim      : str  = format separator pada file csv "," atau ";" etc.
        output_pdf : str  = output file berupa nama file pdf hasil konversi
        col_widths : list = ukuran masing-masing kolom pada file pdf
        output_dir : str  = path folder tujuan penyimpanan (harus sudah ada).
                            jika None, disimpan di folder yang sama dengan file_csv.
    output :
        str : full path file PDF yang dihasilkan
    '''

    # ── Tentukan path output ──────────────────────────────────────────────────
    if output_dir is not None:
        if not os.path.isdir(output_dir):
            raise FileNotFoundError(f"Directory tidak ditemukan: '{output_dir}'")
        output_path = os.path.join(output_dir, os.path.basename(output_pdf))
    else:
        base_dir    = os.path.dirname(os.path.abspath(file_csv))
        output_path = os.path.join(base_dir, os.path.basename(output_pdf))

    # ── Baca CSV ─────────────────────────────────────────────────────────────
    df  = pd.read_csv(file_csv, delimiter=delim).fillna("")
    pdf = FPDF(orientation="L")
    pdf.set_auto_page_break(auto=False)
    pdf.add_page()
    pdf.set_font("helvetica", size=7)

    safe = lambda v: "".join(c if ord(c) < 256 else "?" for c in str(v))
    W    = pdf.w - 2 * pdf.l_margin
    cw   = [w / sum(col_widths) * W for w in col_widths]
    LH   = pdf.font_size * 1.5

    def set_style(bold=False, fill=(255, 255, 255), text=(0, 0, 0)):
        pdf.set_font("helvetica", "B" if bold else "", 7)
        pdf.set_fill_color(*fill)
        pdf.set_text_color(*text)

    def row_h(row):
        max_lines = 1
        for v, w in zip(row, cw):
            text = safe(v)
            if not text:
                continue
            avg_char_w     = pdf.get_string_width("x")
            chars_per_line = max(1, int(w / avg_char_w)) if avg_char_w > 0 else 20
            lines          = math.ceil(len(text) / chars_per_line)
            max_lines      = max(max_lines, lines)
        return max_lines * LH

    def header():
        set_style(bold=True, fill=(52, 73, 94), text=(255, 255, 255))
        pdf.set_x(pdf.l_margin)
        for w, c in zip(cw, df.columns):
            pdf.cell(w, LH, safe(c), border=1, align="C", fill=True)
        pdf.ln()
        set_style()

    header()

    # ── Isi tabel ─────────────────────────────────────────────────────────────
    for i, (_, row) in enumerate(df.iterrows()):
        rh = row_h(row)
        if pdf.get_y() + rh > pdf.h - pdf.b_margin:
            pdf.add_page()
            header()

        y          = pdf.get_y()
        x          = pdf.l_margin
        fill_color = (245, 245, 245) if i % 2 == 0 else (255, 255, 255)
        pdf.set_fill_color(*fill_color)

        for v, w in zip(row, cw):
            pdf.set_xy(x, y)
            pdf.multi_cell(w, LH, safe(v), border=1, fill=True)
            x += w

        pdf.set_xy(pdf.l_margin, y + rh)

    # ── Simpan ke path tujuan ─────────────────────────────────────────────────
    pdf.output(output_path)
    return output_path

In [14]:
csv_to_pdf(
    file_csv="dataMentah.csv",
    delim=";",
    output_pdf="dataMentah.pdf",
    col_widths=[3, 3, 5, 5, 4, 15],  # folder, subfolder, file_name, file_name_stem, id, sentences
)

TypeError: csv_to_pdf() missing 1 required positional argument: 'output_dir'

In [ ]:
csv_to_pdf(
    file_csv="dataClean.csv",
    delim=",",
    output_pdf="dataClean.pdf",
    col_widths=[3, 3, 20, 15],  # folder, subfolder, sentences, path
)

In [ ]:
csv_to_pdf(
    file_csv="smalldataClean.csv",
    delim=",",
    output_pdf="smalldataClean.pdf",
    col_widths=[3, 3, 20, 15],  # folder, subfolder, sentences, path
)

In [ ]:
def pdf_to_csv(input_pdf, output_csv):
    all_rows = []

    with pdfplumber.open(input_pdf) as pdf:
        for page in pdf.pages:
            table = page.extract_table()
            if table:
                for row in table:
                    if row:
                        all_rows.append(row)

    if not all_rows:
        print("Tidak ada tabel ditemukan.")
        return

    header = [str(h) if h is not None else f"col_{i}" for i, h in enumerate(all_rows[0])]
    data   = [row for row in all_rows[1:] if row != all_rows[0]]

    df = pd.DataFrame(data, columns=header)
    df = df.map(lambda x: str(x).strip() if x is not None else "")
    if "sentences" in df.columns:
        df["sentences"] = df["sentences"].str.replace("\n", " ", regex=True)

    df.to_csv(output_csv, index=False)
    print(f"Selesai: {output_csv} ({len(df)} rows, {len(df.columns)} columns)")

In [ ]:
# pdf_to_csv('dataClean.pdf','dataCleanfromPDF.csv')

In [ ]:
# pdf_to_csv('dataMentah.pdf','dataMentahfromPDF.csv')

Bagian ini berfokus pada konversi file FLAC menjadi WAV

In [ ]:
# all_paths = [
#     os.path.abspath(os.path.join(root_dir, file))
#     for root_dir, dirs, files in os.walk(root)
#     for file in files
# ]

In [ ]:
def convert_flac(path_root: str, csv_output: str) -> pd.DataFrame:
    records = []

    for flac_path in glob.glob(os.path.join(path_root, "**/*.flac"), recursive=True):
        try:
            data, sr = sf.read(flac_path)
            wav_path = flac_path.replace(".flac", ".wav")
            sf.write(wav_path, data, sr)

            flac_info = sf.info(flac_path)
            wav_info  = sf.info(wav_path)

            records.append({
                "flac_path"   : flac_path,
                "wav_path"    : wav_path,
                "format_flac" : flac_info.format,
                "format_wav"  : wav_info.format,
                "subtype_flac": flac_info.subtype,
                "subtype_wav" : wav_info.subtype,
                "sr"          : sr,
                "duration_s"  : round(flac_info.duration, 3),
                "size_flac_kb": round(os.path.getsize(flac_path) / 1024, 2),
                "size_wav_kb" : round(os.path.getsize(wav_path)  / 1024, 2),
            })

        except Exception as e:
            print(f"{flac_path}: {e}")

    df = pd.DataFrame(records)
    df.to_csv(csv_output, index=False)
    print(f"\nCSV saved -> {csv_output}")

    return df

In [ ]:
convert_flac(root, 'dataHasilConvert.csv')
convert_flac(sampled2_root, 'smalldataHasilConvert.csv')


CSV saved -> dataHasilConvert.csv

CSV saved -> smalldataHasilConvert.csv


,flac_path,wav_path,format_flac,format_wav,subtype_flac,subtype_wav,sr,duration_s,size_flac_kb,size_wav_kb
0,../data/sampled2\0\00\00079b49f9.flac,../data/sampled2\0\00\00079b49f9.wav,FLAC,WAV,PCM_16,PCM_16,16000,5.400,71.20,168.79
1,../data/sampled2\0\00\002762f6a1.flac,../data/sampled2\0\00\002762f6a1.wav,FLAC,WAV,PCM_16,PCM_16,16000,8.500,159.20,265.67
2,../data/sampled2\0\00\0034ac8d52.flac,../data/sampled2\0\00\0034ac8d52.wav,FLAC,WAV,PCM_16,PCM_16,16000,11.040,265.61,345.04
3,../data/sampled2\0\00\00537fd57f.flac,../data/sampled2\0\00\00537fd57f.wav,FLAC,WAV,PCM_16,PCM_16,16000,4.100,74.89,128.17
4,../data/sampled2\0\00\005a3b3ad4.flac,../data/sampled2\0\00\005a3b3ad4.wav,FLAC,WAV,PCM_16,PCM_16,16000,3.300,71.90,103.17
...,...,...,...,...,...,...,...,...,...,...
1744,../data/sampled2\9\9f\9f6756deab.flac,../data/sampled2\9\9f\9f6756deab.wav,FLAC,WAV,PCM_16,PCM_16,16000,4.500,85.55,140.67
1745,../data/sampled2\9\9f\9f96cbef8d.flac,../data/sampled2\9\9f\9f96cbef8d.wav,FLAC,WAV,PCM_16,PCM_16,16000,4.714,92.50,147.34
1746,../data/sampled2\9\9f\9f9f85a16c.flac,../data/sampled2\9\9f\9f9f85a16c.wav,FLAC,WAV,PCM_16,PCM_16,16000,5.851,115.80,182.90
1747,../data/sampled2\9\9f\9fd1a8c30f.flac,../data/sampled2\9\9f\9fd1a8c30f.wav,FLAC,WAV,PCM_16,PCM_16,16000,4.160,55.59,130.04


In [ ]:
# def convert_all_flac(root: str):
#     for path in glob.glob(os.path.join(root, "**/*.flac"), recursive=True):
#         try:
#             data, sr = sf.read(path)
#             out = path.replace(".flac", ".wav")
#             sf.write(out, data, sr)
#             print(f"{path}..Converted")
#         except Exception as e:
#             print(f"{path}: {e}")

In [ ]:
def clear_flac(path_root: str):
    for path in glob.glob(os.path.join(path_root, "**/*.flac"), recursive=True):
        try:
            os.remove(path)
            print(f"Deleted     : {path}")
        except Exception as e:
            print(f"Error {path}: {e}")

In [ ]:
clear_flac(root)
clear_flac(sampled2_root)

Deleted     : ../data/sampled\0\00\0002114d3f.flac
Deleted     : ../data/sampled\0\00\00079b49f9.flac
Deleted     : ../data/sampled\0\00\00084ff1da.flac
Deleted     : ../data/sampled\0\00\000923633f.flac
Deleted     : ../data/sampled\0\00\0009d31e8f.flac
Deleted     : ../data/sampled\0\00\000e2b3102.flac
Deleted     : ../data/sampled\0\00\000e78988a.flac
Deleted     : ../data/sampled\0\00\00175a0bc1.flac
Deleted     : ../data/sampled\0\00\00197e04b3.flac
Deleted     : ../data/sampled\0\00\001a373938.flac
Deleted     : ../data/sampled\0\00\001dcf14f7.flac
Deleted     : ../data/sampled\0\00\0020c37d51.flac
Deleted     : ../data/sampled\0\00\00210b8a68.flac
Deleted     : ../data/sampled\0\00\0023bd2ccf.flac
Deleted     : ../data/sampled\0\00\002457231d.flac
Deleted     : ../data/sampled\0\00\00246a66f9.flac
Deleted     : ../data/sampled\0\00\002762f6a1.flac
Deleted     : ../data/sampled\0\00\002e271120.flac
Deleted     : ../data/sampled\0\00\0034ac8d52.flac
Deleted     : ../data/sampled\0

In [ ]:
csv_to_pdf(
    file_csv="dataHasilConvert.csv",
    delim=",",
    output_pdf="dataHasilConvert.pdf",
    col_widths=[10, 10, 3,  # flac_path, wav_path, format_flac
                3, 3, 3,    # format_wav, subtype_flac, subtype_wav
                3, 3, 3, 3] # sr, duration_s, size_flac_kb, size_wav_kb             
)

In [ ]:
csv_to_pdf(
    file_csv="smalldataHasilConvert.csv",
    delim=",",
    output_pdf="smalldataHasilConvert.pdf",
    col_widths=[10, 10, 3,  # flac_path, wav_path, format_flac
                3, 3, 3,    # format_wav, subtype_flac, subtype_wav
                3, 3, 3, 3] # sr, duration_s, size_flac_kb, size_wav_kb             
)

In [ ]:
csv_to_pdf(
    file_csv="HasilSampling.csv",
    delim=",",
    output_pdf="HasilSampling.pdf",
    col_widths=[2, 2, 2,  # Wave id ,Sample Rate,	Jumlah Sampel,
                2, 10]  # durasi,Contoh Sample
)

In [ ]:
csv_to_pdf(
    file_csv="HasilQuantization.csv",
    delim=",",
    output_pdf="HasilQuantization.pdf",
    col_widths=[2, 2, 2,  # wav id, Sample Rate, Jumlah Sampel,
                2, 10, 5,  # durasi,contoh sampling, contoh kode PCM,
                10]        # contoh Quantization
)

In [ ]:
csv_to_pdf(
    file_csv="hasilVAD.csv",
    delim=",",
    output_pdf="hasilVAD.pdf",
    col_widths=[2, 2, 2,  #Id,Total Frame,Frame Wicara,
                2, 2, 10]  # Frame Noise,Frame Tdk Pasti,Sinyal Bersih     
)


In [ ]:
def pecah_csv(file, n):
    p = Path(file)
    out = p.parent / f"{p.stem}_pecahan"
    out.mkdir(exist_ok=True)
 
    duplikat = shutil.copy2(file, p.parent / f"{p.stem}_duplikat.csv")
 
    with open(duplikat, encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        baris = list(reader)
 
    per_file = ceil(len(baris) / n)
 
    for i in range(n):
        potongan = baris[i * per_file : (i + 1) * per_file]
        if not potongan:
            break
        nama = out / f"{p.stem}_part{i+1}.csv"
        with open(nama, "w", encoding="utf-8", newline="") as f:
            csv.writer(f).writerows([header] + potongan)
        print(f"{nama.name}={len(potongan):,} baris")
 
    return out

In [ ]:
file = "hasilVAD_detail.csv"
folder = pecah_csv(file, n=10)

for csv_file in sorted(folder.glob(f"{Path(file).stem}_part*.csv")):
    csv_to_pdf(
        file_csv=str(csv_file),
        delim=",",
        output_pdf=str(csv_file.with_suffix(".pdf")),
        col_widths=[4, 2, 4,        # Id, frame_idx, energi
                    4, 4, 4, 4, 4]  # daya, estimasi_daya, snr, snr_db, label
    )

hasilVAD_detail_part1.csv=34,614 baris
hasilVAD_detail_part2.csv=34,614 baris
hasilVAD_detail_part3.csv=34,614 baris
hasilVAD_detail_part4.csv=34,614 baris
hasilVAD_detail_part5.csv=34,614 baris
hasilVAD_detail_part6.csv=34,614 baris
hasilVAD_detail_part7.csv=34,614 baris
hasilVAD_detail_part8.csv=34,614 baris
hasilVAD_detail_part9.csv=34,614 baris
hasilVAD_detail_part10.csv=34,612 baris


In [ ]:
csv_to_pdf(
    file_csv=r'D:\doc\2026\FINAL_CODE\src\codeStat\preModelling\featureExtraction\out_featureExtraction\Data_Feature_Extraction_Beta.csv',
    delim=",",
    output_pdf='Data_Feature_Extraction_Beta.pdf',
    col_widths=[3,                     #audio_id
                3,3,3,3,3,3,3,3,3,3,   #f_0,f_1,f_2,f_3,f_4,f_5,f_6,f_7,f_8,f_9,f_10,
                3,3,3,3,3,3,3,3,3,3,   #f_11,f_12,f_13,f_14,f_15,f_16,f_17,f_18,f_19,f_20,
                3,3,3,3,3,3,3,3,3,3,   #f_21,f_22,f_23,f_24,f_25,f_26,f_27,f_28,f_29,f_30,
                3,3,3,3,3,3,3,3,3],       #f_31,f_32,f_33,f_34,f_35,f_36,f_37,f_38
    output_dir=r'D:\doc\2026\FINAL_CODE\src\codeStat\preModelling\featureExtraction\out_featureExtraction'
)

'D:\\doc\\2026\\FINAL_CODE\\src\\codeStat\\preModelling\\featureExtraction\\out_featureExtraction\\Data_Feature_Extraction_Beta.pdf'